# Oppitunti 08 - Moni-agenttinen suunnittelumalli


## Asennus


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Miksi moniagenttijärjestelmät?

Todelliset tehtävät, kuten matkan suunnittelu, vaativat monenlaista asiantuntemusta — logistiikkaa, paikallistuntemusta, budjetointia ja muuta. Yksittäinen agentti, joka yrittää hoitaa kaiken, muuttuu nopeasti vaikeaksi hallita.

Moniagenttijärjestelmät ratkaisevat tämän **erikoistumisen** kautta: kukin agentti keskittyy yhteen asiantuntemusalueeseen, tuottaen korkealaatuisempia tuloksia kuin yleistyöntekijä. Ne myös parantavat **skaalautuvuutta** — voit lisätä uusia agenteja (esim. lentoasiantuntija, ravintolakriitikko) ilman, että sinun tarvitsee kirjoittaa olemassa olevaa työnkulkua uudelleen. Agentit muodostavat yhdessä rakenteellisen putkiston, välittäen kontekstia yhdeltä toiselle.


## Erikoistuneiden agenttien luominen


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Peräkkäisen työnkulun rakentaminen

`WorkflowBuilder` antaa sinun yhdistää agentteja suuntautuneeseen verkkoon. Tässä luomme yksinkertaisen kahden vaiheen putken: **TravelPlanner** laatii matkasuunnitelman, ja sitten **TravelConcierge** tarkistaa ja parantaa sitä.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Lisää agenteja työnkulkuun

Yksi monen agentin mallin suurimmista eduista on, kuinka helppoa sitä on laajentaa. Alla lisäämme **BudgetReviewer**-agentin, joka tarkistaa suunnitelman matkustajan budjetin näkökulmasta, merkitsee kohteet, jotka saattavat ylittää kustannusrajan, ja ehdottaa säästövinkkejä. Työnkulku suorittaa nyt kolme agenttia peräkkäin:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Yhteenveto

Tällä oppitunnilla opit miten:

1. **Luodaan erikoistuneita agenteja** — jokaisella on oma tarkka roolinsa (suunnittelu, concierge, budjetin tarkastus).
2. **Kytketään agentteja peräkkäiseen työnkulkuun** käyttäen `WorkflowBuilder` ja `add_edge`.
3. **Striimataan output** monen agentin putkistosta, seuraten mikä agentti puhuu.
4. **Laajennetaan työnkulkua** lisäämällä uusia agenteja ketjuun ilman, että olemassa olevia muutetaan.

Moni-agenttinen suunnittelumalli pitää jokaisen agentin yksinkertaisena samalla kun se tuottaa rikkaampia ja perusteellisemmin tarkastettuja tuloksia kuin yksittäinen agentti voisi yksin saavuttaa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:  
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, huomioithan, että automaattikäännöksissä saattaa esiintyä virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäiskielellä tulee pitää virallisena lähteenä. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä johtuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
